## Verifica polizza
_carica_json, verifica_polizza, router_verifica_polizza.


In [ ]:
# ==================== NODO 3 - Edoardo Spinelli - Verifica polizza ====================
# Compito unico: controllare che cliente e polizza esistano nei JSON, che la polizza sia del cliente, NON verifica il budget e i documenti

# Legge i file Json dei clienti e delle polizze, li memorizza in dizionari per accesso rapido, 
# cosi da leggere i dati in unico passaggio e non doverli rileggere ad ogni richiesta.
def _carica_json(nome):
    return json.loads((DATA_DIR / nome).read_text(encoding="utf-8"))


CUSTOMERS = {c["customer_id"]: c for c in _carica_json("customers.json")}
POLICIES  = {p["policy_id"]:   p for p in _carica_json("policies.json")}


# controlla che il clinte e la polizza esistono, che la polizza sia del cliente, che sia attiva e che copra il tipo di sinistro dichiarato
def verifica_polizza(state):
    # riceve in input un dizionario con i dati estratti dal messaggio JSON, restituisce un dizionario con esito (OK o RIFIUTATO) 
    # e in caso di rifiuto la motivazione
    risultato = {}      # dati letti dai JSON, tenuti anche se poi un controllo fallisce

    def rifiuta(motivo):
        # Restituisce un dizionario con esito RIFIUTATO e la motivazione
        return {**risultato, "esito": "RIFIUTATO", "motivazione": motivo}

    # Se l'estrazione ha segnalato dati mancanti, non c'è nulla da verificare
    if state.get("errori"):
        #  se l'estrazione ha segnalato dati mancanti rifiuta subito, prima di leggere i JSON
        return rifiuta("Richiesta incompleta: " + ", ".join(state["errori"]) + ".")

    customer_id, policy_id, loss_type = state["customer_id"], state["policy_id"], state["loss_type"]

    # Il cliente esiste?
    cliente = CUSTOMERS.get(customer_id)
    if not cliente:
        return rifiuta(f"Il cliente {customer_id} non esiste.")
    risultato.update(customer_name=cliente["name"], risk_tier=cliente["risk_tier"])

    # La polizza esiste?
    polizza = POLICIES.get(policy_id)
    if not polizza:
        return rifiuta(f"La polizza {policy_id} non esiste.")

    # La polizza è di QUESTO cliente?
    if polizza["customer_id"] != customer_id:
        # Se non coincidono, blocca la verifica e restituisce un errore di mancata corrispondenza
        return rifiuta(f"La polizza {policy_id} non appartiene al cliente {customer_id}.")

    # Se la polizza è dell'utente, salva tutti i dati rilevanti nel dizionario 'risultato'
    risultato.update(
                product=polizza["product"],
                coverage=polizza["coverage"],
                deductible=float(polizza["deductible"]),
                max_claim=float(polizza["max_claim"]),
                active=bool(polizza["active"]))

    # La polizza è attiva?
    if not polizza["active"]:
        return rifiuta(f"La polizza {policy_id} non è attiva.")

    # La polizza copre il tipo di sinistro dichiarato?
    # --- 5. Verifica Coerenza Copertura ---
    # Accerta che il tipo di evento (loss_type) rientri nella garanzia prevista dal contratto
    if polizza["coverage"] != loss_type:
        return rifiuta(f"La polizza {policy_id} copre '{polizza['coverage']}', non '{loss_type}'.")

    return {**risultato, "esito": "OK"}


def router_verifica_polizza(state):
    return state["esito"]          # OK | RIFIUTATO

# Log di avvenuta inizializzazione del nodo
print("✅ Nodo verifica_polizza definito —", len(CUSTOMERS), "clienti,", len(POLICIES), "polizze")

## Analizza i documenti

ontiene i documenti validi, i requisiti, il nodo analizza_documenti, il router e il driver esegui_pratica. Gli import interrupt e Command sono in questo file e servono anche al Nodo 7.

In [ ]:
# ==================== NODO 4 - Analizza documenti (loop + interrupt) ====================
# Compito unico: controllare che il cliente abbia allegato tutti i documenti richiesti

from langgraph.types import interrupt, Command

# Tipi di documento formalmente riconosciuti dal sistema
DOCUMENTI_VALIDI = {"photo", "police_report", "repair_invoice"}

# Regole per tipo di sinistro (documenti richiesti + soglia di approvazione automatica)
# Carica dal JSON i documenti obbligatori e le soglie per ciascun tipo di sinistro (es. 'water_leak')
REQUIREMENTS = {r["loss_type"]: r for r in _carica_json("requirements.json")}


def analizza_documenti(state):
    # Estrae le regole specifiche per il danno
    requisiti = REQUIREMENTS[state["loss_type"]]       # KeyError esplicito se il loss_type non è previsto
    richiesti = state.get("required_documents") or requisiti["required_documents"]
    soglia    = float(requisiti["automatic_review_max"])   # serve a verifica_budget
    presenti  = list(state.get("documents") or [])  # Lista di documenti già caricati
    retry     = state.get("retry_count", 0)         # Contatore dei tentativi effettuati

    # Funzione interna ausiliaria per calcolare l'insieme dei file che ancora mancano
    def mancanti():
        return [d for d in richiesti if d not in presenti]

    # Già tutto in regola
    # Caso 1: La documentazione è già completa fin da subito
    if not mancanti():
        return {"required_documents": richiesti, "automatic_review_max": soglia,
                "documents": presenti, "esito": "OK"}

    # Tentativi esauriti
    # Caso 2: Sono stati già esauriti i tentativi a disposizione
    if retry >= MAX_RETRY:
        return {
            "required_documents": richiesti,
            "automatic_review_max": soglia,
            "esito": "TROPPI_TENTATIVI",
            "motivazione": f"Documenti mancanti dopo {MAX_RETRY} tentativi: {', '.join(mancanti())}."
            }

    # Pausa: il grafo si ferma e restituisce questo payload a chi lo ha invocato.
    # Alla ripresa (Command(resume=...)) interrupt() ritorna il valore scritto dall'utente.
    
    # Caso 3: Mancano dei documenti -> Sospensione del flusso di lavoro
    # 'interrupt' blocca il grafo inviando il dict/payload all'esterno
    risposta = interrupt({
        "messaggio":    "Documenti mancanti per la pratica. Inserisci un documento da allegare.",
        "mancanti":     mancanti(),
        "validi":       sorted(DOCUMENTI_VALIDI),
        "tentativo":    retry + 1,
        "max_tentativi": MAX_RETRY,
    })

    # Ripresa dell'esecuzione: si analizza il dato inserito dall'utente (Command(resume=...))
    doc = str(risposta).strip().lower()
    if doc in DOCUMENTI_VALIDI and doc not in presenti:
        presenti.append(doc)
    # Incrementa il contatore tentativi ad ogni risposta ricevuta
    retry += 1          # ogni interrupt conta come un tentativo, anche se l'input non è valido

    aggiornamenti = {"required_documents": richiesti, 
                     "automatic_review_max": soglia,
                     "documents": presenti, 
                     "retry_count": retry
    }

    # Ricontrolla se con il nuovo inserimento i documenti sono diventati completi o no
    if not mancanti():
        return {**aggiornamenti, "esito": "OK"}
    if retry >= MAX_RETRY:
        return {**aggiornamenti, "esito": "TROPPI_TENTATIVI",
                "motivazione": f"Documenti mancanti dopo {MAX_RETRY} tentativi: {', '.join(mancanti())}."}

    # Se mancano ancora file e ci sono altri tentativi disponibili
    return {**aggiornamenti, "esito": "AMBIGUO"}


def router_analizza_documenti(state):

    # Instrada il flusso del grafo verso il nodo successivo in base all'esito
    return state["esito"]          # OK | AMBIGUO | TROPPI_TENTATIVI


def esegui_pratica(richiesta: str, thread_id: str = "pratica-1"):
    """Driver: lancia il grafo e, a ogni interrupt, chiede l'input da tastiera (documento o decisione del perito)."""

    # Configurazione della memoria della sessione (obbligatoria per salvataggi con MemorySaver)
    config = {"configurable": {"thread_id": thread_id}}      # obbligatorio con MemorySaver

    # Lancia l'esecuzione del grafo
    risultato = graph.invoke({"richiesta": richiesta}, config)

    # Ciclo di gestione degli interruzionali (Human-in-the-Loop)
    while "__interrupt__" in risultato:
        info = risultato["__interrupt__"][0].value

        # RAMO A: Se l'interruzione proviene dal nodo 'chiama_perito'
        if info.get("tipo") == "perito":
            print(f"\n🧑‍⚖️ {info['messaggio']}")
            print(f"   Report:\n{info['report']}")
            print(f"   Pratica {info['claim_id']} ({info['customer_name']}) - {info['loss_type']}")
            print(f"   Danno stimato: {info['estimated_damage']:.2f} €  |  soglia automatica: "
                  f"{info['automatic_review_max']:.2f} €  |  massimale: {info['max_claim']:.2f} €")
            risposta = input("   Il perito approva? (si/no): ")

        # RAMO B: Se l'interruzione proviene dal nodo 'analizza_documenti'
        else:
            print(f"\n📎 {info['messaggio']}")
            print(f"   Mancanti: {', '.join(info['mancanti'])}  |  Validi: {', '.join(info['validi'])}")
            print(f"   Tentativo {info['tentativo']}/{info['max_tentativi']}")
            risposta = input("   Documento da allegare: ")

        # Riprende il grafo inviando l'input dell'utente al punto esatto dell'interrupt
        risultato = graph.invoke(Command(resume=risposta), config)

    # Quando non ci sono più interruzioni, restituisce l'esito finale della pratica
    return risultato


print("✅ Nodo analizza_documenti definito")

## claims_review

l file contiene: PROMPT_REVIEW, _fatti_review, _report_standard, _report_valido, claims_review e, in fondo, un elenco di «MODIFICHE DA FARE» nelle altre celle.


In [ ]:
# ==================== NODO 6 - Edoardo Spinelli - Chiama perito ====================
# Compito unico: sul ramo ESCALATION, scrivere un report descrittivo della pratica per il perito.
# Legge SOLO dallo StatoRichiesta (nessun file esterno): il report non valuta e non decide.

PROMPT_REVIEW = """
    Sei il Claims Review Agent di una compagnia assicurativa. Scrivi un report interno che riassume
    una pratica di sinistro per il perito.
    Ricevi i FATTI già verificati dal sistema. Regole:
    - Scrivi in italiano, tono professionale e neutro, 4-6 frasi, senza elenchi e senza titoli.
    - Descrivi i fatti: prodotto e tipo di sinistro, danno stimato, soglia automatica, di quanto la supera,
      massimale, franchigia, percentuale del danno sul massimale, documenti presenti e rischio cliente.
    - Il rischio cliente (risk_tier) va sempre riportato, con il valore scritto nei fatti: non interpretarlo
      e non dire se sia alto o basso.
    - Usa SOLO i fatti ricevuti: non inventare importi, precedenti o circostanze.
    - Riporta gli importi e le percentuali esattamente come sono scritti nei fatti.
    - Non dare giudizi e non consigliare nulla: niente pareri su approvare o rifiutare,
      la decisione non è tua e viene presa da altri.
    - Non usare codici (es. AS001) al posto del nome del cliente.
    - Non serve firmarti.
    """


def _fatti_review(state):
    """Solo i dati utili al perito, letti dallo stato; percentuali e differenze calcolate dal codice."""

    # Converte i valori numerici principali dallo 'state' in numeri decimali (float)
    danno, max_claim = float(state["estimated_damage"]), float(state["max_claim"])
    soglia = float(state["automatic_review_max"])

    # Recupera le liste dei documenti (se non esistono nello stato, usa una lista vuota [])
    presenti = state.get("documents") or []
    richiesti = state.get("required_documents") or []

    # Costruisce il dizionario con la sintesi dei dati per il perito
    fatti = {
        "pratica":                state.get("claim_id", "?"),   # ID della pratica (o '?' se manca)
        "prodotto":               state.get("product", "?"),
        "tipo_sinistro":          TIPI_SINISTRO.get(state.get("loss_type"), state.get("loss_type", "?")),

        "rischio_cliente":        state.get("risk_tier", "n/d"),        # Fascia di rischio del cliente
        "danno_stimato":          _euro(danno),                         # Importo del danno formattato in €
        "soglia_automatica":      _euro(soglia),                        # Soglia oltre la quale serve il perito
        "superamento_soglia":     _euro(danno - soglia),                # Di quanto il danno supera la soglia
        "massimale":              _euro(max_claim),                     # Massimale massimo coperto
        "franchigia":             _euro(state.get("deductible", 0)),    # Franchigia a carico del cliente

        # Calcola la percentuale del danno rispetti al massimale (es. "75%")
        "danno_su_massimale":     f"{danno / max_claim * 100:.0f}%" if max_claim else "n/d",    

        "documenti_presenti":     presenti,     # Elenco dei file inviati

        # Trova i documenti richiesti che NON figurano tra quelli inviati
        "documenti_mancanti":     [d for d in richiesti if d not in presenti],
    }

    # Se presente nel contesto, aggiunge anche il nome completo del cliente
    if state.get("customer_name"):
        fatti["nome_cliente"] = state["customer_name"]
    return fatti


def _report_standard(fatti):
    """Report di riserva, senza LLM."""

    # Estrae la lista dei documenti mancanti per comodità
    mancanti = fatti["documenti_mancanti"]

    # Assembla un testo formattato inserendo tutti i dettagli tecnici e finanziari
    return (f"Pratica {fatti['pratica']} ({fatti['prodotto']}, {fatti['tipo_sinistro']}): "
            f"rischio cliente {fatti['rischio_cliente']}. Danno stimato {fatti['danno_stimato']}, "
            f"sopra la soglia automatica ({fatti['soglia_automatica']}) di {fatti['superamento_soglia']} "
            f"e pari al {fatti['danno_su_massimale']} del massimale ({fatti['massimale']}). "
            f"Franchigia {fatti['franchigia']}. "
            f"Documenti presenti: {', '.join(fatti['documenti_presenti']) or 'nessuno'}; "
            f"mancanti: {', '.join(mancanti) if mancanti else 'nessuno'}.")


def _report_valido(testo, fatti):

    # 1. Se il testo generato è vuoto o None, non è valido
    if not testo:
        return False

    # 2. Verifica che la parte intera dell'importo del danno (es. '1500' da '1500,00 €') sia presente nel testo
    if fatti["danno_stimato"].split(",")[0] not in testo:               # l'importo (parte intera) deve comparire
        return False

    # 3. Verifica che il livello di rischio del cliente sia citato nel testo
    rischio = str(fatti["rischio_cliente"]).lower()
    if rischio != "n/d" and rischio not in testo.lower():               # il risk_tier deve comparire
        return False

    # 4. Controlla che il testo NON contenga parole come 'consiglio' o 'raccomando' (il perito deve decidere in autonomia)
    if re.search(r"\b(consiglio|raccomand)\w*", testo, flags=re.IGNORECASE):   # niente verdetti
        return False

    # Se passa tutti i controlli, il report generato dall'LLM è considerato valido
    return True


def claims_review(state):

    # 1. Prepara il dizionario 'fatti' con tutti i dati necessari e formattati
    fatti = _fatti_review(state)

    # 2. Prepara i messaggi da inviare all'LLM (istruzioni di sistema + dati della pratica in JSON)
    messaggi = [SystemMessage(content=PROMPT_REVIEW),
                HumanMessage(content="FATTI:\n" + json.dumps(fatti, ensure_ascii=False, indent=2))]

    # 3. Invoca il modello di intelligenza artificiale per generare il report
    testo = llm.invoke(messaggi).content

    # 4. Pulisce l'output rimuovendo i blocchi <think>...</think> (se l'LLM usa il ragionamento 'CoT')
    testo = re.sub(r"<think>.*?</think>", "", testo, flags=re.DOTALL).strip()

    # 5. Valida la risposta dell'LLM: se invalida o non affidabile, usa il report di riserva (fallback)
    if not _report_valido(testo, fatti):
        print("⚠️ claims_review: risposta del modello non affidabile, uso il report standard")
        testo = _report_standard(fatti)

    # 6. Restituisce il report finale aggiornando lo stato della pipeline
    return {"report_perito": testo}


print("✅ Nodo claims_review definito")


# ==================== MODIFICHE DA FARE NELLE ALTRE CELLE ====================
#
# 1) StatoRichiesta: aggiungere il campo
#        report_perito:          str    # report del Claims Review Agent, letto dal perito
#
# 2) chiama_perito: aggiungere al payload dell'interrupt
#        "report": state.get("report_perito", "Nessun report disponibile."),
#    e in esegui_pratica, nel ramo `if info.get("tipo") == "perito":`, aggiungere
#        print(f"   Report:\n{info['report']}")
#
# 3) Grafo:
#        builder.add_node("claims_review", claims_review)
#        builder.add_edge("claims_review", "chiama_perito")
#    e nel conditional edge di verifica_budget cambiare:
#        "ESCALATION": "claims_review"      # prima era "chiama_perito"

## chiama al perito

chiama_perito2.py. Contiene RISPOSTE_SI, RISPOSTE_NO, chiama_perito e router_decisione_perito. 

In [ ]:
# ==================== NODO 7 - Edoardo Spinelli - Chiama perito ====================
# Compito unico: sottoporre al perito una pratica in ESCALATION e raccogliere il suo sì/no

# Set di stringhe valide per interpretare la risposta del perito (sia in italiano che in inglese)
RISPOSTE_SI = {"si", "sì", "s", "yes", "y"}
RISPOSTE_NO = {"no", "n"}


def chiama_perito(state):

    # 1. Costruisce il pacchetto di dati (payload) da mostrare all'interfaccia del perito
    payload = {
        "tipo":                 "perito",
        "messaggio":            "Pratica oltre la soglia di approvazione automatica: serve la decisione del perito.",
        "claim_id":             state.get("claim_id", "?"),
        "customer_name":        state.get("customer_name", "?"),
        "loss_type":            state.get("loss_type", "?"),
        "estimated_damage":     float(state["estimated_damage"]),
        "automatic_review_max": float(state["automatic_review_max"]),
        "max_claim":            float(state["max_claim"]),
        "documents":            state.get("documents", []),
        "report": state.get("report_perito", "Nessun report disponibile.")
    }

    # Se la risposta non è né sì né no, richiedo: la decisione del perito non va mai indovinata.
    # 2. Ciclo di attesa/validazione: garantisce che la decisione sia univoca e non ambigua
    while True:
        # Sospende l'esecuzione di LangGraph/workflow e invia il payload all'utente;
        # riprende quando l'utente risponde e assegna il valore alla variabile 'risposta'
        risposta = interrupt(payload)

        # Normalizza la risposta ricevuta (rimuove spazi e rende minuscolo)
        scelta = str(risposta).strip().lower()

        # Se la risposta è affermativa
        if scelta in RISPOSTE_SI:
            return {"esito": "APPROVATO",
                    "motivazione": f"Il perito ha approvato la pratica (danno stimato {payload['estimated_damage']:.2f} €)."}

        # Se la risposta è negativa
        if scelta in RISPOSTE_NO:
            return {"esito": "RIFIUTATO",
                    "motivazione": f"Il perito ha rifiutato la pratica (danno stimato {payload['estimated_damage']:.2f} €)."}

        payload = {**payload, "messaggio": f"Risposta '{risposta}' non valida: rispondi 'si' oppure 'no'."}


def router_decisione_perito(state):
    # Router del grafo che legge lo stato finale e instrada il flusso verso il nodo successivo in base all'esito
    return state["esito"]          # APPROVATO | RIFIUTATO


print("✅ Nodo chiama_perito definito")